# Gate C4 v2_1 — Requalification Launcher

**This notebook contains no scientific logic. It only invokes tested scripts.**

Every step of the run lives in `scripts/`, is imported by the test suite,
and is covered by pytest. That is deliberate: the previous notebooks were
independent implementations, drifted from the tested code path, and were
fail-open where the protocol requires an abort. They are preserved under
`notebooks/superseded/` for provenance and must not be used.

```
one implementation  ->  scripts/colab_c4_requalify.py
        v
tested in pytest    ->  tests/unit/test_c4_*.py
        v
notebook merely invokes it
```

**Prerequisites:** Runtime -> Change runtime type -> **T4 GPU + High-RAM**.
Expected wall time: ~30-40 minutes.

If you want to change what the run does, edit the script and add a test.
Do not add logic to this notebook.


## 1. Check out the exact revision

Set `EXPECTED_COMMIT` to the **full SHA** of the revision you intend to
certify. That revision must already contain a *captured* environment lock
(see section 3) — a lock with `null` pins aborts the run at step 3.

The notebook clones and checks out; the script verifies. The script no
longer clones anything, so a session cannot end up holding two checkouts.


In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/dawsonblock/Daph-ex-research-gate-c2-beir-retrieval.git"
PROJECT_ROOT = Path("/content/repo")

# The full SHA of the revision to certify. Fill this in.
EXPECTED_COMMIT = ""  # e.g. "db0e9b335ba0..."

assert EXPECTED_COMMIT, "Set EXPECTED_COMMIT to the revision you intend to certify"

if not PROJECT_ROOT.exists():
    !git clone {REPO_URL} {PROJECT_ROOT}

os.chdir(PROJECT_ROOT)
!git fetch --all --quiet
!git checkout --quiet {EXPECTED_COMMIT}
os.environ["PYTHONPATH"] = str(PROJECT_ROOT)

!git rev-parse HEAD
print("cwd:", os.getcwd())

### Optional: mount Drive

Only needed if you want the result bundle written somewhere durable.


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# OUTPUT_DIR = Path('/content/drive/MyDrive/c4_results')
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Confirm HEAD and a clean source tree

The run aborts unless `HEAD == EXPECTED_COMMIT` and nothing under
`hrm_adaptive_memory/`, `scripts/`, `configs/`, `tests/`, `daph/` or
`pyproject.toml` is modified.

Changes under `evidence/` are fine — the run writes there itself.


In [ ]:
!git rev-parse HEAD
!git status --porcelain

from hrm_adaptive_memory.c4 import git_state
state = git_state.inspect(Path.cwd())
print("HEAD matches:", git_state.revision_matches(state.head, EXPECTED_COMMIT))
print("source clean:", state.source_clean)
print("evidence changes (expected):", len(state.output_changes))

## 3. Freeze the environment

Run this **before** the requalification, in the runtime that will produce
the result. The lock committed in the repository holds transcribed values
with `null` pins and fails certification by design — an unrecorded version
is a failure, not a wildcard.

After this writes a concrete lock, restart the runtime if practical, then
re-run cells 1-2 and continue.


In [ ]:
!python scripts/c4_freeze_environment.py --note 'colab T4 certifying run'

In [ ]:
# Verify the live environment against the lock before spending GPU time.
!python scripts/c4_freeze_environment.py --check

## 4. Run the fail-closed requalification

This is the whole experiment. It aborts on any protocol abort condition
(test failure, determinism failure, prompt-binding violation), runs the
primary ladder `C4_0..C4_6` plus the diagnostic arms `C4_3o`/`C4_4m`,
analyses, and certifies.


In [ ]:
!python scripts/colab_c4_requalify.py --expected-commit {EXPECTED_COMMIT}

## 5. Read the certificate

`VALID_RUN` is the conjunction of every derived gate. Only if it is `true`
may the bundle be cited as a conformant C4 v2_1 result.


In [ ]:
import json

cert_path = Path("evidence/gate_c4/full/development/certification/CERTIFICATION.json")
cert = json.loads(cert_path.read_text())
print("VALID_RUN:", cert["VALID_RUN"])
print("verdict:  ", cert["verdict"])
print("gates:    ", cert["gates_passed"], "/", cert["gates_total"])
for name in cert["gates_failed"]:
    print("  FAILED:", name)
    for v in cert["gates"][name]["violations"][:5]:
        print("    -", v)

In [ ]:
# Ordering vs membership 2x2 — the question the rerun exists to answer.
dec = cert["performance_summary"]["ordering_membership_decomposition"] or {}
if not dec.get("available"):
    print("decomposition unavailable:", dec.get("missing_arms"))
else:
    q = dec["quality"]
    print(f"Q(C4_3)  = {q['C4_3']:.4f}   S0  membership + pool order")
    print(f"Q(C4_3o) = {q['C4_3o']:.4f}   S0  membership + deterministic order")
    print(f"Q(C4_4m) = {q['C4_4m']:.4f}   S2c membership + pool order")
    print(f"Q(C4_4)  = {q['C4_4']:.4f}   S2c membership + deterministic order")
    print()
    print(f"E_order_S0        = {dec['ordering_effect']:+.4f}")
    print(f"E_membership_pool = {dec['membership_effect']:+.4f}")
    print(f"E_order_S2c       = {q['C4_4'] - q['C4_4m']:+.4f}")
    print(f"interaction       = {dec['interaction_effect']:+.4f}")

## 6. Verify hashes last


In [ ]:
!cd evidence/gate_c4/full/development && sha256sum -c RESULTS.sha256

---

**Stop before the qualification split.** Development authorising
qualification is a separate decision, made against the D1-D8 promotion
gates once `VALID_RUN` is `true`.

`C4_3o` and `C4_4m` are diagnostic. They explain `C4_4`; they are not
candidates for promotion and do not enter the primary ladder or the
promotion threshold.
